In [ ]:
!pip install piexif

Forgery Tests

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image, ImageChops, ImageEnhance
import piexif
import json
from io import BytesIO
from google.colab import files

# Function to check image quality
def check_image_quality(image):
    """
    Checks image quality based on Laplacian variance to determine blurriness.

    Args:
        image (np.ndarray): Image as a NumPy array.

    Returns:
        bool: True if image is sharp (low variance), False if it is blurry (high variance).
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance > 100  # Threshold for blurriness

# Function to check color consistency
def check_color_consistency(image):
    """
    Checks for large color standard deviation potentially indicating manipulation.

    Args:
        image (np.ndarray): Image as a NumPy array.

    Returns:
        bool: True if color standard deviation is high, False otherwise.
    """
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mean, stddev = cv2.meanStdDev(hsv_image)
    return stddev[1] < 50  # Arbitrary threshold for color consistency

# Function to detect steganography
def detect_steganography(image):
    """
    Detects potential steganography by analyzing LSB of the image.

    Args:
        image (np.ndarray): Image as a NumPy array.

    Returns:
        bool: True if LSB analysis indicates potential steganography, False otherwise.
    """
    red_channel_lsb = image[:, :, 2] & 1  # LSB of the red channel
    green_channel_lsb = image[:, :, 1] & 1  # LSB of the green channel
    blue_channel_lsb = image[:, :, 0] & 1  # LSB of the blue channel

    # Check for consistent LSB values across channels
    if np.all(red_channel_lsb == green_channel_lsb) and np.all(red_channel_lsb == blue_channel_lsb):
        return True  # Steganography not detected
    else:
        return False  # Steganography detected

# Function to check noise consistency
def check_noise_consistency(image):
    """
    Checks for consistent noise pattern indicating image integrity.

    Args:
        image (np.ndarray): Image as a NumPy array.

    Returns:
        bool: True if noise pattern is consistent, False otherwise.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

    # Calculate standard deviation of Laplacian
    stddev = laplacian.std()

    # Adjust threshold based on image size and quality
    threshold = 10  # Adjust this value based on your testing and image characteristics

    return stddev < threshold

# Function to compress image using JPEG
def compress_image(image, quality=90):
    """
    Compresses the image using JPEG compression.

    Args:
        image (PIL.Image.Image): The input image.
        quality (int): The JPEG compression quality (default: 90).

    Returns:
        PIL.Image.Image: The compressed image.
    """
    output = BytesIO()
    image.save(output, format='JPEG', quality=quality)
    output.seek(0)
    compressed_image = Image.open(output)
    return compressed_image

# Function to perform ELA
def perform_ela(image_path):
    """
    Performs basic Error Level Analysis (ELA) on an image.

    Args:
        image_path (str): Path to the image file.

    Returns:
        PIL.Image: The ELA image.
        float: The ELA score (placeholder, replace with actual calculation).
        dict: Additional ELA result information (optional).
    """
    # Load the image
    original = Image.open(image_path)
    original = original.convert('RGB')  # Ensure it's in RGB mode
    resaved_path = '/content/temp_resaved.jpg'
    original.save(resaved_path, 'JPEG', quality=90)

    resaved = Image.open(resaved_path)

    # Calculate the difference
    ela_image = ImageChops.difference(original, resaved)

    # Enhance the difference to make it more visible
    extrema = ela_image.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    scale = 255.0 / max_diff
    ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)

    # Convert to numpy array for further analysis
    ela_array = np.array(ela_image)
    ela_score = np.mean(ela_array)  # Placeholder for actual ELA score calculation

    # Additional ELA results (optional)
    ela_result = {"score": ela_score}  # You can add more information here

    return ela_image, ela_score, ela_result

# Function to check metadata
from PIL import Image
import piexif

def check_metadata(image_path):
    """
    Checks the metadata of the image for anomalies.

    Args:
        image_path (str): Path to the image file.

    Returns:
        dict: Dictionary containing metadata analysis result.
    """
    try:
        image = Image.open(image_path)
        exif_data = image.info.get('exif')
        if exif_data:
            exif_dict = piexif.load(exif_data)
            software_used = find_photo_editing_software(exif_dict)
            if software_used:
                return {
                    "metadata_result": False,
                    "software_used": ", ".join(software_used),
                    "metadata_reason": "Edited by photo editing software"
                }
            else:
                return {
                    "metadata_result": True
                }
        else:
            return {
                "metadata_result": True
            }
    except Exception as e:
        print(f"Error checking metadata: {e}")
        return {
            "metadata_result": False,
            "metadata_reason": str(e)
        }

def find_photo_editing_software(exif_dict):
    """
    Finds photo editing software in EXIF data.

    Args:
        exif_dict (dict): Dictionary containing EXIF data.

    Returns:
        list: List of photo editing software found in EXIF data.
    """
    editing_software_keywords = ["photoshop", "gimp", "paintshop", "corel", "lightroom", "luminar", "snapseed", "picmonkey"]
    software_used = []

    if piexif.ImageIFD.Software in exif_dict.get('0th', {}):
        software = exif_dict['0th'][piexif.ImageIFD.Software].decode("utf-8").lower()
        for keyword in editing_software_keywords:
            if keyword in software:
                software_used.append(keyword.capitalize())

    return software_used

# Function to export forensic analysis results to JSON
def export_to_json(metadata_result, ela_score, image_quality, color_consistency, steganography_detected, noise_consistency, output_folder):
    """
    Exports forensic analysis results to a JSON file.

    Args:
        metadata_result (dict): Dictionary containing metadata analysis result.
        ela_score (float): ELA score.
        image_quality (bool): Image quality check result.
        color_consistency (bool): Color consistency check result.
        steganography_detected (bool): Whether steganography is detected.
        noise_consistency (bool): Noise consistency check result.
        output_folder (str): Path to the output folder.
    """
    # Convert numpy bool to Python bool
    metadata_result["metadata_result"] = bool(metadata_result["metadata_result"])

    # Flip steganography_detected result
    steganography_detected = not steganography_detected

    result_dict = {
        "metadata_result": metadata_result["metadata_result"],
        "software_used": metadata_result.get("software_used", ""),
        "metadata_reason": metadata_result.get("metadata_reason", ""),
        "ela_score": ela_score,
        "image_quality": bool(image_quality),  # Convert image_quality to Python bool
        "color_consistency": bool(color_consistency),  # Convert color_consistency to Python bool
        "steganography": steganography_detected,  # Include steganography detection result
        "noise_consistency": bool(noise_consistency)  # Convert noise_consistency to Python bool
    }

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    json_file = os.path.join(output_folder, 'Forensics.json')
    with open(json_file, 'w') as f:
        json.dump(result_dict, f, indent=4)

    print(f"Forensics result exported to {json_file}")

# Function to handle file upload and forensic analysis
def handle_image_upload_and_analysis():
    # Upload image file
    uploaded = files.upload()
    image_filename = list(uploaded.keys())[0]
    image_path = image_filename

    # Load the image using OpenCV
    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Could not load image from {image_path}. Please check the file path and ensure it's a valid image.")
    else:
        try:
            # Perform ELA
            ela_image, ela_score, ela_result = perform_ela(image_path)

            # Perform forensic checks
            image_quality = check_image_quality(image)
            color_consistency = check_color_consistency(image)
            steganography_detected = detect_steganography(image)
            noise_consistency = check_noise_consistency(image)

            # Check metadata
            metadata_result = check_metadata(image_path)

            # Export results to JSON
            output_folder = '/content/Forensics'
            export_to_json(metadata_result, ela_score, image_quality, color_consistency, steganography_detected, noise_consistency, output_folder)

            # Print additional results
        except Exception as e:
            print(f"Error processing image: {e}")

# Example usage
if __name__ == "__main__":
    handle_image_upload_and_analysis()

Saving 1000364152E.jpg to 1000364152E.jpg
Forensics result exported to /content/Forensics/Forensics.json


Forgery Report

In [ ]:
import json

def count_false_and_check_ela(json_file):
    """
    Counts the number of occurrences of "False" in boolean keys of a JSON file
    and prints locations of "False" values and "ELA is high" if necessary.

    Args:
        json_file (str): Path to the JSON file.

    Returns:
        int: Number of "False" occurrences.
    """
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)

        count_false = 0
        false_locations = []

        # Iterate over each key in the JSON data
        for key, value in data.items():
            if isinstance(value, bool):
                if value is False:
                    count_false += 1
                    false_locations.append(key)

        # Check ELA score
        ela_score = data.get('ela_score', 0.0)
        if ela_score > 5.5:
            print(f"ELA is high: {ela_score}")

        if count_false > 0:
            print(f"Number of Test failed: {count_false}")
            print(f"The Tests: {false_locations}")

        return count_false

    except FileNotFoundError:
        print(f"Error: File '{json_file}' not found.")
    except json.JSONDecodeError:
        print(f"Error: JSON format error in file '{json_file}'.")
    except Exception as e:
        print(f"Error: {e}")

    return 0

# Example usage
if __name__ == "__main__":
    json_file_path = '/content/Forensics/Forensics.json'  # Replace with your JSON file path
    false_count = count_false_and_check_ela(json_file_path)


ELA is high: 6.2562080866280425
Number of Test failed: 3
The Tests: ['metadata_result', 'color_consistency', 'noise_consistency']


Report File

In [ ]:
import json

# Define the data for the JSON file
data = {
    "metadata_result": True,
    "software_used": "",
    "metadata_reason": "",
    "ela_score": "Less than 5.5",  # Use quotes for string values
    "image_quality": True,
    "color_consistency": True,
    "steganography": True,
    "noise_consistency": True
}

# Write the data to a JSON file named "Valid_Forensics.json"
with open('/content/Forensics/Valid_Forensics.json', 'w') as outfile:
    json.dump(data, outfile, indent=4)

print("Valid_Forensics.json generated successfully!")

Valid_Forensics.json generated successfully!


Photo Deletion

In [ ]:
import os

def delete_image_files(folder_path):
    # Define common image file extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}

    # Iterate over all files in the folder
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # Checks if the file has one of the image extensions
        if os.path.isfile(file_path) and os.path.splitext(filename)[1].lower() in image_extensions:
            os.remove(file_path)
            print(f'Deleted: {file_path}')

# Example usage
folder_path = '/content/'
delete_image_files(folder_path)
